In [ ]:
"""
Concepts:

1. Gradient Boosting
2. Learning Rate
3. Boosting
4. Regularization
5. Early Stopping
6. Feature Importance
7. Permutation Importance
8. Learning Curves
"""
# =====================================================
# IMPORTS
# =====================================================

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import (train_test_split,cross_val_score,GridSearchCV,RandomizedSearchCV,learning_curve)
from sklearn.metrics import (mean_absolute_error,mean_squared_error,r2_score)
from sklearn.inspection import permutation_importance

In [ ]:
# =====================================================
# STEP 1 : DATA UNDERSTANDING
# =====================================================

housing = fetch_california_housing()
df = pd.DataFrame(housing.data,columns=housing.feature_names)
df["Target"] = housing.target

print(df.head())
print(df.info())
print(df.describe())

In [ ]:
# =====================================================
# STEP 2 : EDA
# =====================================================

print(df.isnull().sum())
print(df.duplicated().sum())
print(df.corr())

In [ ]:
# =====================================================
# STEP 3 : TRAIN / VALIDATION / TEST
# =====================================================

X = df.drop("Target", axis=1)
y = df["Target"]
X_temp, X_test, y_temp, y_test = train_test_split(X,y,test_size=0.15,random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp,y_temp,test_size=0.1765,random_state=42)

# =====================================================
# STEP 4 : PREPROCESSING
# =====================================================

# Trees do not require scaling


In [ ]:
# =====================================================
# STEP 5 : BASELINE MODEL
# =====================================================

xgb = XGBRegressor(
    n_estimators=100,   learning_rate=0.1,
    max_depth=6,    min_child_weight=1, subsample=1.0,
    colsample_bytree=1.0,   gamma=0,    reg_alpha=0,
    reg_lambda=1,   objective="reg:squarederror",   random_state=42
)

xgb.fit(X_train,y_train)

In [ ]:
# =====================================================
# STEP 6 : VALIDATION METRICS with base model
# =====================================================

y_pred = xgb.predict(X_val)
mae = mean_absolute_error(y_val,y_pred)
rmse = np.sqrt(mean_squared_error(y_val,y_pred))
r2 = r2_score(y_val,y_pred)

print("\nBASELINE TREE")
print("MAE :", mae)
print("RMSE:", rmse)
print("R2 :", r2)

In [ ]:
# =====================================================
# STEP 7 : CROSS VALIDATION
# =====================================================
cv_scores = cross_val_score(xgb,X_train,y_train,cv=5,scoring="r2",n_jobs=-1)
print("\nCV Mean:", cv_scores.mean())
print("CV Std :", cv_scores.std())

In [ ]:
# =====================================================
# STEP 8 : RANDOMIZED SEARCH
# =====================================================

# XGBoost parameter space is huge.
# RandomizedSearchCV is preferred.

param_grid = {
    "n_estimators":[100,200,300],
    "learning_rate":[0.01,0.05,0.1,0.2],
    "max_depth":[3,4,5,6],
    "subsample":[0.8,1.0],
    "colsample_bytree":[0.8,1.0],
    "gamma":[0,0.1,0.3]}

random_search = RandomizedSearchCV(estimator=XGBRegressor(objective="reg:squarederror",random_state=42),
    param_distributions=param_grid,n_iter=20,cv=3,scoring="r2",random_state=42,n_jobs=-1)

random_search.fit(X_train,y_train)
print(random_search.best_params_)
best_model = random_search.best_estimator_

In [ ]:
# =====================================================
# STEP 9 : VALIDATION METRICS with best model
# =====================================================

y_best = best_model.predict(X_val)
val_r2 = r2_score(y_val,y_best)
print("Validation R2:", val_r2)

In [ ]:
# =====================================================
# STEP 10 : TRAIN VS VALIDATION
# =====================================================

y_train_best = best_model.predict(X_train)
train_r2 = r2_score(y_train,y_train_best)
print("Train R2:", train_r2)
print("Validation R2:", val_r2)

In [ ]:
# =====================================================
# STEP 11 : XGBOOST ANALYSIS
# =====================================================

# ---------------------------------------------
# Feature Importance
# ---------------------------------------------

importance_df = pd.DataFrame({"Feature":X.columns,"Importance":best_model.feature_importances_}).sort_values(by="Importance",ascending=False)
plt.figure(figsize=(8,5))
plt.barh(importance_df["Feature"],importance_df["Importance"])
plt.title("Feature Importance")
plt.show()

# ---------------------------------------------
# Permutation Importance
# ---------------------------------------------

perm = permutation_importance(best_model,X_val,y_val,random_state=42)
perm_df = pd.DataFrame({"Feature":X.columns,"Importance":perm.importances_mean})
print(perm_df)

# ---------------------------------------------
# Learning Rate Analysis
# ---------------------------------------------

learning_rates = [0.01,0.05,0.1,0.2,0.3]

scores = []

for lr in learning_rates:
    model = XGBRegressor(learning_rate=lr,n_estimators=200,random_state=42,objective="reg:squarederror")
    model.fit(X_train,y_train)
    pred = model.predict(X_val)
    score = r2_score(y_val,pred)
    scores.append(score)

plt.plot(learning_rates,scores,marker="o")
plt.title("Learning Rate Analysis")
plt.xlabel("Learning Rate")
plt.ylabel("Validation R2")
plt.show()

# ---------------------------------------------
# Learning Curve
# ---------------------------------------------

train_sizes, train_scores, val_scores = learning_curve(best_model, X_train,y_train,cv=5,scoring="r2",n_jobs=-1)

plt.plot(train_sizes,np.mean(train_scores, axis=1),label="Train")
plt.plot(train_sizes,np.mean(val_scores, axis=1),label="Validation")
plt.legend()
plt.title("Learning Curve")
plt.show()


In [ ]:
# =====================================================
# STEP 12 : FINAL MODEL
# =====================================================

final_model = best_model

In [ ]:
# =====================================================
# STEP 13 : TEST EVALUATION
# =====================================================

test_pred = final_model.predict(X_test)
test_mae = mean_absolute_error(y_test,test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test,test_pred))
test_r2 = r2_score(y_test,test_pred)

print("\nTEST RESULTS")
print("MAE :", test_mae)
print("RMSE:", test_rmse)
print("R2 :", test_r2)

In [ ]:
# =====================================================
# STEP 14 : INTERPRETATION
# =====================================================

print("\nTOP FEATURES")
print(importance_df.head(10))

In [ ]:
# =====================================================
# STEP 15 : DEPLOYMENT READINESS
# =====================================================

print("\nDEPLOYMENT READINESS")

print("CV Mean:",cv_scores.mean())
print("Validation R2:",val_r2)
print("Test R2:",test_r2)


# =====================================================
# INTERVIEW QUESTIONS
# =====================================================

"""
1. What is Boosting?
2. Difference between Bagging and Boosting?
3. Why XGBoost beats Random Forest?
4. What is learning_rate?
5. What is gamma?
6. What is subsample?
7. What is colsample_bytree?
8. What is reg_alpha?
9. What is reg_lambda?
10. What is min_child_weight?
11. Why does XGBoost prevent overfitting?
12. Why is XGBoost fast?
13. Why does XGBoost handle missing values?
14. GridSearchCV vs RandomizedSearchCV?
15. What is early stopping?
16. What is feature importance?
17. What is permutation importance?
18. When would Random Forest beat XGBoost?
19. What are disadvantages of XGBoost?
20. Explain XGBoost from scratch.
"""